In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as ctx
import numpy as np
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.colors import LogNorm
from shapely import wkt
from matplotlib.patches import FancyArrowPatch

In [ ]:
class MapVisualization:
    def __init__(self):
        # Define Swiss cities coordinates (EPSG:3857)
        self.swiss_cities = {
            'Zürich': [950857.6943765444, 6003812.2040421115],
            'Geneva': [683857.8957460445, 5813165.184246248],
            'Basel': [844759.0877162443, 6033898.2105036415],
            'Bern': [829040.7756184222, 5933590.474586056],
            'Lausanne': [738348.7864817666, 5864026.255820999],
            'Lucerne': [924987.044719789, 5950271.842115855],
            'Winterthur': [974045.5443055555, 6024072.118535226],
            'St. Gallen': [1043809.4691759888, 6011509.009002252],
            'Lugano': [996431.8939009666, 5780942.166871505],
            'Biel': [806843.6691573333, 5964431.969528809],
            'Sion': [816339.2217206778, 5817814.854684204],
            'Thun': [849156.2076019666, 5902467.595356518],
            'Chur': [1061164.1777882446, 5917461.7627768945]
        }
        self.gdf = None



    def load_data(self, file_path):
        """
        Load and prepare geodataframe from CSV file
        
        """
        df = pd.read_csv(file_path)
        df['geometry'] = df['geometry'].apply(wkt.loads)
        self.gdf = gpd.GeoDataFrame(df, crs='EPSG:4326')
        return self.gdf
    
    def add_switzerland_border(self, ax, border_color='black', border_width=1, alpha=0.8):
        """
        Add Switzerland country border to the map
        
        """
        border_file = "switzerland.geojson"
        ch_border = gpd.read_file(border_file)
            
            # Convert to EPSG:3857 if not already in this projection
        if ch_border.crs != 'EPSG:3857':
            ch_border = ch_border.to_crs(epsg=3857)
        
        # Plot the border
        ch_border.boundary.plot(
            ax=ax,
            color=border_color,
            linewidth=border_width,
            alpha=alpha,
            zorder=4  # Make sure it's drawn on top of other layers
        )
        

    def add_north_arrow(self, ax, position='upper right'):
        """
        Add a north arrow to the map

        """
        # Get the current axis limits
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        
        # Calculate the position for the north arrow
        if position == 'upper right':
            arrow_x = xlim[1] - (xlim[1] - xlim[0]) * 0.1
            arrow_y = ylim[1] - (ylim[1] - ylim[0]) * 0.1
        elif position == 'upper left':
            arrow_x = xlim[0] + (xlim[1] - xlim[0]) * 0.1
            arrow_y = ylim[1] - (ylim[1] - ylim[0]) * 0.1
        
        # Create the arrow
        arrow_length = (ylim[1] - ylim[0]) * 0.05
        arrow = FancyArrowPatch(
            (arrow_x, arrow_y - arrow_length),
            (arrow_x, arrow_y + arrow_length),
            arrowstyle='simple',
            color='black',
            mutation_scale=15
        )
        ax.add_patch(arrow)
        
        # Add "N" label
        ax.text(
            arrow_x,
            arrow_y + arrow_length * 1.2,
            'N',
            horizontalalignment='center',
            verticalalignment='bottom',
            fontsize=12,
            fontweight='bold'
        )
    

    def add_scale_bar(self, ax, position='lower left'):
        """
        Add a scale bar to the map
        
        """
        # Get the current axis limits
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        
        # Calculate the scale bar length (in meters)
        # For Switzerland, we'll use a 10km scale bar
        scale_length = 10000  # 10 km in meters
        
        # Calculate the position for the scale bar
        if position == 'lower left':
            bar_x = xlim[0] + (xlim[1] - xlim[0]) * 0.1
            bar_y = ylim[0] + (ylim[1] - ylim[0]) * 0.1
        elif position == 'lower right':
            bar_x = xlim[1] - (xlim[1] - xlim[0]) * 0.1 - scale_length
            bar_y = ylim[0] + (ylim[1] - ylim[0]) * 0.1
        
        # Draw the scale bar
        ax.plot([bar_x, bar_x + scale_length], [bar_y, bar_y], 'k-', linewidth=2, color='black')
        ax.plot([bar_x, bar_x], [bar_y - scale_length/50, bar_y + scale_length/50], 'k-', linewidth=2,color='black')
        ax.plot([bar_x + scale_length, bar_x + scale_length], 
                [bar_y - scale_length/50, bar_y + scale_length/50], 'k-', linewidth=2,color='black')
        
        # Add label
        ax.text(
            bar_x + scale_length/2,
            bar_y + scale_length/25,
            '10 km',
            horizontalalignment='center',
            verticalalignment='bottom',
            fontsize=10,
            fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )




    def plot_location_visit_heatmap(self, title='Synthetic Location Visit Frequency'):
        """
        Create a heatmap of location visits

        """
        if self.gdf is None:
            raise ValueError("No data loaded. Please call load_data() first.")

        # Group by location and calculate counts
        gdf_grouped = self.gdf.groupby('location_id').size().reset_index(name='count')
        gdf_corresponding = self.gdf[['location_id', 'geometry']].drop_duplicates()
        gdf_grouped = gdf_grouped.merge(gdf_corresponding, on='location_id')
        gdf_grouped = gpd.GeoDataFrame(gdf_grouped, crs='EPSG:4326')
        gdf_plot = gdf_grouped.to_crs(epsg=3857)

        # Create figure and axis
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))

        # Calculate color scale bounds
        # vmin = np.percentile(gdf_plot['count'], 5)
        # vmax = np.percentile(gdf_plot['count'], 95)
        vmin = 10**1
        vmax = 10**3.2

        # Plot the heatmap
        gdf_plot.plot(
            column='count',
            ax=ax,
            legend=True,
            legend_kwds={
                'orientation': 'vertical',
                'shrink': 0.3,
                'norm': LogNorm(vmin=vmin, vmax=vmax)
            },
            cmap='YlOrRd',
            norm=LogNorm(vmin=vmin, vmax=vmax),
            alpha=0.5,
            markersize=50
        )

        # Add city markers and labels
        for city, coords in self.swiss_cities.items():
            ax.plot(coords[0], coords[1], 'k*', markersize=12, zorder=5)
            ax.annotate(
                city,
                xy=(coords[0], coords[1]),
                xytext=(5, 5),
                textcoords='offset points',
                fontsize=10,
                fontweight='bold',
                color='black',
            )

        # Set plot bounds with padding
        bounds = gdf_plot.total_bounds
        padding = (bounds[2] - bounds[0]) * 0.1
        ax.set_xlim(bounds[0] - padding, bounds[2] + padding)
        ax.set_ylim(bounds[1] - padding, bounds[3] + padding)

        # Add OSM basemap
        ctx.add_basemap(
            ax,
            source=ctx.providers.CartoDB.Positron,
            zoom=8
        )

        # Add north arrow and scale bar
        self.add_north_arrow(ax, position='upper right')
        self.add_scale_bar(ax, position='lower left')
        self.add_switzerland_border(ax, border_color='black', border_width=1.5, alpha=0.8)

        # Customize the plot
        ax.set_title(title, pad=20, fontsize=16)
        ax.set_axis_off()

        # Adjust layout
        plt.tight_layout()
        
        return fig, ax


    def plot_user_trajectory(self, user_id=None, top_n_labels=5, model_name=None):
        """
        Plot the trajectory of a user, with point sizes indicating visit frequency
        and labels on the most visited locations.

        Args:
            user_id (int, optional): The ID of the user to plot. Defaults to the first user in the dataset.
            top_n_labels (int, optional): The number of top-visited locations to label with percentages. Defaults to 5.
        """
        if self.gdf is None:
            raise ValueError("No data loaded. Please call load_data() first.")

        # If no user_id is provided, use the first one in the dataset
        if user_id is None:
            user_id = self.gdf['user_id'].unique()[0]

        # Filter data for the selected user and sort by sequence
        user_data = self.gdf[self.gdf['user_id'] == user_id].copy()
        if user_data.empty:
            print(f"Warning: No data found for user ID {user_id}")
            return None, None
        user_data = user_data.sort_values('sequence')

        # --- 1. Calculate visit counts and percentages for this user ---
        location_counts = user_data['location_id'].value_counts()
        total_visits = len(user_data)

        # Get the unique locations to plot them as single points
        unique_locations = user_data.drop_duplicates(subset='location_id').copy()
        unique_locations['visit_count'] = unique_locations['location_id'].map(location_counts)
        unique_locations['visit_percentage'] = (unique_locations['visit_count'] / total_visits) * 100

        # Convert to EPSG:3857 for web mercator projection
        user_data_mercator = user_data.to_crs(epsg=3857)
        unique_locations_mercator = unique_locations.to_crs(epsg=3857)

        # --- 2. Create the plot ---
        fig, ax = plt.subplots(1, 1, figsize=(12, 10))

        # --- 3. Plot the trajectory line (the grey "web") ---
        coords = [(point.x, point.y) for point in user_data_mercator.geometry]
        path = Path(coords)
        patch = PathPatch(
            path,
            facecolor='none',
            edgecolor='lightblue',  # Use a subtle color for the path
            linewidth=1.5,
            alpha=0.8,
            zorder=2  # Place it below the points
        )
        ax.add_patch(patch)

        # --- 4. Plot the location points with scaled sizes ---
        # Define a scaling range for marker sizes. You can adjust these values.
        min_marker_size = 10
        max_marker_size = 150 # A large max size creates a dramatic effect

        # Calculate the size for each unique location based on its visit count
        max_count = location_counts.max()
        if max_count > 0:
            unique_locations_mercator['marker_size'] = min_marker_size + \
                (unique_locations_mercator['visit_count'] / max_count) * (max_marker_size - min_marker_size)
        else:
            unique_locations_mercator['marker_size'] = min_marker_size

        # Plot the unique locations
        unique_locations_mercator.plot(
            ax=ax,
            marker='o',
            markersize=unique_locations_mercator['marker_size'],
            column='location_id',  # Color each location differently
            # cmap='tab20',          # A colormap with many distinct colors
            color='green',
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=3               # Ensure points are on top of the path
        )

        total_gdf_mercator = self.gdf.to_crs(epsg=3857)
        bounds = total_gdf_mercator.total_bounds
        
        padding_x = (bounds[2] - bounds[0]) * 0.05
        padding_y = (bounds[3] - bounds[1]) * 0.05
        
        ax.set_xlim(bounds[0] - padding_x, bounds[2] + padding_x)
        ax.set_ylim(bounds[1] - padding_y, bounds[3] + padding_y)

        # Add basemap and other map elements
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=9, zorder=1)
        self.add_switzerland_border(ax, border_color='black', border_width=1.5, alpha=0.8)
        self.add_north_arrow(ax, position='upper right')
        self.add_scale_bar(ax, position='lower left')

        # Customize the plot
        ax.set_title(f'User No.{user_id} Trajectory ({model_name})', pad=20, fontsize=16)
        ax.set_axis_off()


        plt.tight_layout()
        return fig, ax



    def plot_user_visit_heatmap(self, user_id=None, title='user visit heatmap'):
        """
        Create a heatmap of user location visits
        """
        if self.gdf is None:
            raise ValueError("No data loaded. Please call load_data() first.")
        
        # If no user_id provided, use the first one in the dataset
        if user_id is None:
            user_id = self.gdf['user_id'].unique()[0]
        
        # Filter data for the selected user
        user_data = self.gdf[self.gdf['user_id'] == user_id].copy()
        
        if len(user_data) == 0:
            raise ValueError(f"No data found for user ID {user_id}")
        
        # Group by location and calculate visit counts
        location_counts = user_data.groupby('location_id').size().reset_index(name='count')
        
        # Merge with location geometries
        location_geometries = user_data[['location_id', 'geometry']].drop_duplicates()
        user_locations = location_counts.merge(location_geometries, on='location_id')
        user_locations = gpd.GeoDataFrame(user_locations, crs='EPSG:4326')
        
        # Convert to Web Mercator for visualization
        user_locations = user_locations.to_crs(epsg=3857)
        
        # Create figure and axis
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        
        # Calculate color scale bounds
        # vmin = user_locations['count'].min()
        # vmax = user_locations['count'].max()
            # Calculate color scale bounds
        vmin = np.percentile(user_locations['count'], 0)
        vmax = np.percentile(user_locations['count'], 98)
        # Plot the heatmap
        user_locations.plot(
            column='count',
            ax=ax,
            legend=True,
            legend_kwds={
                'orientation': 'vertical',
                'shrink': 0.3,
                'label': 'Visit Count',
                'norm': LogNorm(vmin=vmin, vmax=vmax)
            },
            norm=LogNorm(vmin=vmin, vmax=vmax),
            cmap='YlOrRd',
            alpha=0.7,
            markersize=100
        )
        
        # Add city markers and labels
        for city, coords in self.swiss_cities.items():
            ax.plot(coords[0], coords[1], 'k*', markersize=12, zorder=5)
            ax.annotate(
                city,
                xy=(coords[0], coords[1]),
                xytext=(5, 5),
                textcoords='offset points',
                fontsize=10,
                fontweight='bold',
                color='black',
            )
        
        # Set plot bounds with padding
        bounds = user_locations.total_bounds
        padding = (bounds[2] - bounds[0]) * 0.1
        ax.set_xlim(bounds[0] - padding, bounds[2] + padding)
        ax.set_ylim(bounds[1] - padding, bounds[3] + padding)
        
        # Add OSM basemap
        ctx.add_basemap(
            ax,
            source=ctx.providers.CartoDB.Positron,
            zoom=8
        )
        
        # Add north arrow and scale bar
        self.add_north_arrow(ax, position='upper right')
        self.add_scale_bar(ax, position='lower left')
        self.add_switzerland_border(ax, border_color='black', border_width=1.5, alpha=0.8)
        
        # Customize the plot
        if title is None:
            title = f'User No.{user_id} Location Visit Frequency'
        ax.set_title(title, pad=20, fontsize=16)
        ax.set_axis_off()
        
        # Adjust layout
        plt.tight_layout()
        
        return fig, ax


In [ ]:
#instantiate the class
map = MapVisualization()
# Load data
gdf = map.load_data('../output/ipt.csv')

In [ ]:
fig, ax = map.plot_location_visit_heatmap(title='Location Visit Frequency(EPR)')
plt.show()

In [ ]:
fig, ax = map.plot_user_trajectory(user_id= 200, model_name='IPT')
plt.show()

In [ ]:
fig, ax = map.plot_user_visit_heatmap(user_id= 600)
plt.show()